# 00 - DRC Validation Map

Synthetic engineering and validation evidence, not final regulatory capital.

Map the DRC notebook pack to the committed non-securitisation fixture, public API, and regulatory anchors.

Use this notebook as a companion to [`PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) and [`examples/run_demo.py`](../examples/run_demo.py). The happy path is the public package API: construct DRC positions and a calculation context, then call `frtb_drc.calculate_drc_capital`.

## Raw inputs your upstream risk engine must emit

A DRC upstream engine should emit one row per default-risk position with stable position and source-row identifiers, desk/legal-entity scope, risk class, instrument type, issuer or tranche/index identifiers where applicable, bucket key, seniority or credit quality, notional or market value, maturity, currency, lineage fields, and citation identifiers. The non-securitisation table contract is documented in [`docs/schemas/input_table/frtb_drc.nonsec.schema.json`](../../../docs/schemas/input_table/frtb_drc.nonsec.schema.json); package-specific context and fixture notes live in [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)

## Public API happy path

The primary user contract is public API first: pass validated `DrcPosition` records and `DrcCalculationContext` to `calculate_drc_capital`. The implementation-detail sections below inspect intermediate mechanics for validation and auditability.


In [ ]:
from frtb_drc import calculate_drc_capital

public_result = calculate_drc_capital(positions, context=context)
print(f"Total DRC capital: {public_result.total_drc:,.2f}")
print(f"Input hash       : {public_result.input_hash}")
print(f"Profile hash     : {public_result.profile_hash}")


## Notebook map

In [ ]:
rows = [
    ["00_validation_map.ipynb", "Pack index and regulatory framework", "positions.json, expected_outputs.json"],
    ["01_gross_jtd.ipynb", "LGD rules and gross JTD calculation", "positions.json, expected_outputs.json"],
    ["02_maturity_scaling.ipynb", "Maturity weighting floor and ramp", "positions.json, expected_outputs.json"],
    ["03_netting.ipynb", "Same-obligor netting and seniority constraints", "positions.json, expected_outputs.json"],
    ["04_hbr_bucket_capital.ipynb", "Hedge benefit ratio and bucket capital", "positions.json, expected_outputs.json"],
    ["05_category_capital.ipynb", "Category total and multi-desk analysis", "positions.json, expected_outputs.json"],
]
display(markdown_table(["Notebook", "Purpose", "Fixture inputs"], rows))

## Regulatory anchors

US NPR 2.0 (91 FR 14952) is the primary profile.  Basel FRTB (MAR22) is the
conceptual baseline.

| Step | Rule ref | Description |
|------|----------|-------------|
| Gross JTD | § 210(b)(1)(iv) / MAR22.11-13 | LGD * notional + P&L component |
| Maturity weight | § 210(a)(2)(iii) / MAR22.12 | Floor 0.25 Y, linear ramp to 1.0 Y |
| Netting | § 210(b)(2) / MAR22.14 | Same obligor, seniority constraint |
| HBR | § 210(a)(2)(iv)(A) / MAR22.17 | agg\_long / (agg\_long + agg\_short) |
| Bucket capital | § 210(a)(2)(iv)(C) / MAR22.19 | Σ(RW * long) - HBR * Σ(RW * short), ≥ 0 |
| Category total | § 210(b)(3)(iii) / MAR22.20 | Sum of bucket capitals |

## LGD rules (US NPR 2.0)

In [ ]:
from frtb_drc.reference_data import iter_lgd_rules, US_NPR_2_0_PROFILE_ID

rows = [
    [rule.seniority.value, f"{rule.lgd_rate:.0%}", rule.citation_id, rule.description]
    for rule in iter_lgd_rules(US_NPR_2_0_PROFILE_ID)
]
display(markdown_table(["Seniority", "LGD", "Citation", "Description"], rows))

## Risk weights by bucket and credit quality (US NPR 2.0)

In [ ]:
from frtb_drc.reference_data import iter_risk_weight_rules

rows = [
    [rule.bucket_key, rule.credit_quality.value, f"{rule.risk_weight:.1%}", rule.citation_id]
    for rule in iter_risk_weight_rules(US_NPR_2_0_PROFILE_ID)
]
display(markdown_table(["Bucket", "Credit quality", "Risk weight", "Citation"], rows))

## Portfolio summary

In [ ]:
from collections import Counter

by_bucket = Counter(p.bucket_key for p in positions)
by_desk = Counter(p.desk_id for p in positions)
by_cq = Counter(p.credit_quality for p in positions)
by_dir = Counter(p.default_direction for p in positions)

print(f"Total positions : {len(positions)}")
print(f"By bucket       : {dict(sorted(by_bucket.items()))}")
print(f"By desk         : {dict(sorted(by_desk.items()))}")
print(f"By credit qual  : {dict(sorted(by_cq.items()))}")
print(f"By direction    : {dict(sorted(by_dir.items()))}")
print(f"Total DRC (USD) : {expected['total_drc']:,.2f}")

## See also

- [`packages/frtb-drc/docs/PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) for the package workflow and maturity path.
- [`packages/frtb-drc/docs/DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md) for fixture and input-shape notes.
- [`docs/CLIENT_INTEGRATION.md`](../../../docs/CLIENT_INTEGRATION.md) for suite-level client integration guidance.
- Sibling component journeys: [`frtb-sbm`](../../frtb-sbm/docs/PACKAGE_JOURNEY.md), [`frtb-rrao`](../../frtb-rrao/docs/PACKAGE_JOURNEY.md), [`frtb-cva`](../../frtb-cva/docs/PACKAGE_JOURNEY.md), [`frtb-ima`](../../frtb-ima/docs/PACKAGE_JOURNEY.md), and [`frtb-orchestration`](../../frtb-orchestration/docs/PACKAGE_JOURNEY.md).
